<a href="https://colab.research.google.com/github/yuzxin/Capstone/blob/main/ASOS_%EC%A0%84%EC%B2%98%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import numpy as np
import pandas as pd

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 데이터 파일 불러오기

In [28]:
# 1. 데이터 파일 로드 (한글 깨짐 방지를 위해 cp949 인코딩 사용)
file_path = "★OBS_ASOS_DD_mean_min_max.csv"
df = pd.read_csv(file_path, encoding="cp949")

# 2. 원본 컬럼명을 데이터 설명서 기준의 영문 표준 명칭으로 강제 변환
df.columns = [
    "Station_ID",  # 지점(관측소ID)
    "Location",  # 지점명(관측소명)
    "Date",  # 일시(관측일자)
    "Temp_Avg",  # 평균기온
    "Temp_Min",  # 최저기온
    "Temp_Max",  # 최고기온
]

# 3. 영어로 컬럼명이 잘 바뀌었는지 상위 5줄 확인
df.head()

,Station_ID,Location,Date,Temp_Avg,Temp_Min,Temp_Max
0,90,속초,2016-01-01,3.6,-2.3,7.6
1,90,속초,2016-01-02,8.4,5.0,11.5
2,90,속초,2016-01-03,6.7,2.4,9.8
3,90,속초,2016-01-04,5.4,0.8,9.0
4,90,속초,2016-01-05,0.6,-2.6,4.4


### 날짜 데이터 형식 변환 (시계열)

In [31]:
# 1. 문자열 상태인 '일시' 컬럼을 판다스 날짜 객체(datetime)로 변환
df["Date"] = pd.to_datetime(df["Date"])

In [32]:
# 2. 연도, 월, 일 컴포넌트 분리 생성
df["연도"] = df["Date"].dt.year
df["월"] = df["Date"].dt.month
df["일"] = df["Date"].dt.day

# 3. 1월 1일부터 몇 번째 날짜인지 알려주는 '경과일(1~365)' 생성 (적산온도 시각화용)
df["경과일"] = df["Date"].dt.dayofyear

# 날짜 변환이 잘 되었는지 데이터 형태 재확인
df[["Date", "연도", "월", "일", "경과일"]].head()

,Date,연도,월,일,경과일
0,2016-01-01,2016,1,1,1
1,2016-01-02,2016,1,2,2
2,2016-01-03,2016,1,3,3
3,2016-01-04,2016,1,4,4
4,2016-01-05,2016,1,5,5


### 결측치 탐지 및 시계열 보정

In [34]:
# 1. 전처리 전 컬럼별 결측치(NaN) 개수 확인 출력
print("--- 전처리 전 빈 데이터 개수 ---")
print(df.isnull().sum())

# 2. 기후 흐름을 깨지 않도록 '지점별(Location)'로 묶은 후, 비어있는 기온을 '직전 날짜 기온(ffill)'으로 1차 보정
df["Temp_Avg"] = df.groupby("Location")["Temp_Avg"].ffill()
df["Temp_Min"] = df.groupby("Location")["Temp_Min"].ffill()
df["Temp_Max"] = df.groupby("Location")["Temp_Max"].ffill()

# 3. 만약 첫날부터 데이터가 비어있어 안 채워진 경우 '바로 다음 날 기온(bfill)'으로 2차 보정
df["Temp_Avg"] = df.groupby("Location")["Temp_Avg"].bfill()
df["Temp_Min"] = df.groupby("Location")["Temp_Min"].bfill()
df["Temp_Max"] = df.groupby("Location")["Temp_Max"].bfill()

print("\n--- 전처리 후 빈 데이터 개수 (모두 0이어야 함) ---")
print(df.isnull().sum())

--- 전처리 전 빈 데이터 개수 ---
Station_ID      0
Location        0
Date            0
Temp_Avg      376
Temp_Min       33
Temp_Max       27
연도              0
월               0
일               0
경과일             0
dtype: int64

--- 전처리 후 빈 데이터 개수 (모두 0이어야 함) ---
Station_ID    0
Location      0
Date          0
Temp_Avg      0
Temp_Min      0
Temp_Max      0
연도            0
월             0
일             0
경과일           0
dtype: int64


### 데이터 논리 오류 정제 (기온 모순 데이터 이상치 필터링)

In [36]:
# 1. 논리적으로 말이 안 되는 이상치 데이터가 몇 건이나 있는지 확인
invalid_rows = df[df["Temp_Min"] > df["Temp_Max"]]
print(f"물리적 기온 오류 데이터 개수: {len(invalid_rows)}건")

물리적 기온 오류 데이터 개수: 0건


In [39]:
# 2. 최저기온이 최고기온보다 같거나 낮은 정상적인 데이터만 남기고 필터링
df = df[df["Temp_Min"] <= df["Temp_Max"]]
print(f"이상치 정제 후 남은 총 데이터 개수: {len(df):,}건")

이상치 정제 후 남은 총 데이터 개수: 348,749건


### 분석용 파생 변수 생성 (일교차 및 적산온도)

In [41]:
# 1. 일교차 생성 (최고기온 - 최저기온)
# 일교차 컬럼명 = Temp_Range
df["Temp_Range"] = df["Temp_Max"] - df["Temp_Min"]

In [46]:
# # 밀원수 개화 관측용 파생 변수 (적산온도) 계산

# 1. 누적 적산온도 계산을 위해 지점명(Location)과 일시(Date) 기준으로 완벽하게 정렬
df = df.sort_values(by=["Location", "Date"]).reset_index(drop=True)

# [팁] 연도별(Year)로 묶어서 계산하기 위해, Date(YYYY-MM-DD) 컬럼에서 앞 4자리 연도만 쏙 뽑아 임시 컬럼을 만듭니다.
df["Year"] = df["Date"].astype(str).str[:4]

# 2. 식물생태학 표준 공식 변형: 밀원수 기준온도인 10도 이하인 날은 0점 처리, 10도 초과인 날만 '기온 - 10도' 점수 부여
# 유효기온 컬럼명 = Effective_Temp
df["Effective_Temp"] = df["Temp_Avg"].apply(lambda x: (x - 10) if x > 10 else 0)

# 3. 10도를 빼고 남은 진짜 알짜배기 성장 점수('Effective_Temp')를 지점별, 연도별로 묶어서 누적합(cumsum) 수행
# 적산온도 컬럼명 = Accumulated_Temp
df["Accumulated_Temp"] = df.groupby(["Location", "Year"])["Effective_Temp"].cumsum()

# 4. 계산용 임시 컬럼(연도, 유효기온) 제거 및 최종 생성 결과 확인
df = df.drop(columns=["Year", "Effective_Temp"])

# 5. 최종 변경된 영어 컬럼명으로 데이터 상위 5줄 출력 확인
df[["Location", "Date", "Temp_Range", "Accumulated_Temp"]].head()

,Location,Date,Temp_Range,Accumulated_Temp
0,강릉,2016-01-01,6.8,0.0
1,강릉,2016-01-02,5.7,0.0
2,강릉,2016-01-03,8.5,0.0
3,강릉,2016-01-04,8.7,0.0
4,강릉,2016-01-05,7.3,0.0


### 정제된 마스터 파일 내보내기

In [47]:
# 1. 전처리가 완료된 데이터프레임의 최종 요약 정보 확인
print("--- 전처리 완료 데이터 구조 최종 확인 ---")
print(df.info())

# 2. 깨끗해진 데이터를 'cleaned_ASOS_weather_data.csv' 파일로 저장 (인코딩 보정 포함)
output_file = "cleaned_ASOS_weather_data.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(
    f"\n 전처리가 완료되어 파일이 '{output_file}'(으)로 보관됨"
)

--- 전처리 완료 데이터 구조 최종 확인 ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348749 entries, 0 to 348748
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Station_ID        348749 non-null  int64         
 1   Location          348749 non-null  object        
 2   Date              348749 non-null  datetime64[ns]
 3   Temp_Avg          348749 non-null  float64       
 4   Temp_Min          348749 non-null  float64       
 5   Temp_Max          348749 non-null  float64       
 6   연도                348749 non-null  int32         
 7   월                 348749 non-null  int32         
 8   일                 348749 non-null  int32         
 9   경과일               348749 non-null  int32         
 10  Temp_Range        348749 non-null  float64       
 11  Accumulated_Temp  348749 non-null  float64       
dtypes: datetime64[ns](1), float64(5), int32(4), int64(1), object(1)
memory usage: 26.6+ MB

In [50]:
# 1. 데이터가 날짜순으로 꼬이지 않도록 'Location(지점명)'과 'Date(일시)' 기준으로 완벽하게 정렬
df_sorted = df.sort_values(["Location", "Date"])

# [팁] 연도별 누적합 계산을 위해, Date(YYYY-MM-DD) 컬럼에서 앞 4자리 연도만 쏙 뽑아 임시 컬럼을 만듭니다.
df_sorted["Year"] = df_sorted["Date"].astype(str).str[:4]

# 2. groupby('Location', 'Year')를 쓰면, 상위 몇 개가 아니라
#    데이터에 존재하는 "전국 전체 지점"을 연도별로 각각 묶어서 누적합(cumsum)을 계산합니다.
#    (0도 기준이므로 Temp_Avg를 그대로 누적해서 더합니다.)
df_sorted["Accumulated_Temp"] = df_sorted.groupby(["Location", "Year"])[
    "Temp_Avg"
].cumsum()

# [추가] 계산용으로 썼던 임시 연도(Year) 컬럼은 깔끔하게 삭제합니다.
df_sorted = df_sorted.drop(columns=["Year"])

# 3. 누적 계산이 올바르게 전 지점에 적용되었는지 샘플 확인 (상위 5행)
print("--- 전처리 완료 샘플 데이터 확인 (영문 컬럼명 반영) ---")
display(
    df_sorted[["Location", "Date", "Temp_Avg", "Temp_Range", "Accumulated_Temp"]].head()
)

# 4. 전체 지점의 적산온도가 포함된 최종 마스터 파일을 CSV로 내보내기
#    한글 깨짐 방지를 위해 encoding="utf-8-sig"를 추가했습니다.
df_sorted.to_csv("적산온도_기준0도.csv", index=False, encoding="utf-8-sig")
print("\n전국 전체 지점의 적산온도(기준 0도) 컬럼 추가 및 CSV 저장 완료!")

--- 전처리 완료 샘플 데이터 확인 (영문 컬럼명 반영) ---


,Location,Date,Temp_Avg,Temp_Range,Accumulated_Temp
0,강릉,2016-01-01,5.1,6.8,5.1
1,강릉,2016-01-02,9.1,5.7,14.2
2,강릉,2016-01-03,9.0,8.5,23.2
3,강릉,2016-01-04,7.4,8.7,30.6
4,강릉,2016-01-05,3.0,7.3,33.6



전국 전체 지점의 적산온도(기준 0도) 컬럼 추가 및 CSV 저장 완료!


In [49]:
# 1. 데이터가 날짜순으로 꼬이지 않도록 'Location(지점명)'과 'Date(일시)' 기준으로 완벽하게 정렬
df_sorted = df.sort_values(["Location", "Date"])

# [팁] 연도별 누적합 계산을 위해, Date(YYYY-MM-DD) 컬럼에서 앞 4자리 연도만 쏙 뽑아 임시 컬럼을 만듭니다.
df_sorted["Year"] = df_sorted["Date"].astype(str).str[:4]

# [추가] 매일의 평균기온(Temp_Avg)에서 기준온도 10도를 빼줍니다. 단, 10도 미만이라 마이너스가 나오면 0점으로 처리합니다.
df_sorted["Effective_Temp"] = df_sorted["Temp_Avg"].apply(
    lambda x: (x - 10) if x > 10 else 0
)

# 2. 10도를 빼고 남은 진짜 성장 점수('Effective_Temp')를 연도별/지점별로 묶어서 누적합(cumsum)을 계산합니다.
#    groupby('Location', 'Year')를 쓰면 데이터에 존재하는 "전국 전체 지점"을 연도별로 각각 묶어서 계산합니다.
df_sorted["Accumulated_Temp"] = df_sorted.groupby(["Location", "Year"])[
    "Effective_Temp"
].cumsum()

# [추가] 계산용으로 썼던 임시 컬럼(Year, Effective_Temp)들은 보고서에 필요 없으니 깔끔하게 삭제합니다.
df_sorted = df_sorted.drop(columns=["Year", "Effective_Temp"])

# 3. 누적 계산이 올바르게 전 지점에 적용되었는지 샘플 확인 (상위 5행)
print("--- 전처리 완료 샘플 데이터 확인 (영문 컬럼명 및 기준온도 10도 반영) ---")
display(
    df_sorted[["Location", "Date", "Temp_Avg", "Temp_Range", "Accumulated_Temp"]].head()
)

# 4. 전체 지점의 적산온도가 포함된 최종 마스터 파일을 CSV로 내보내기
#    한글 깨짐 방지를 위해 encoding="utf-8-sig"를 추가했습니다.
df_sorted.to_csv("적산온도_기준10도.csv", index=False, encoding="utf-8-sig")
print("\n전국 전체 지점의 적산온도(기준 10도) 컬럼 추가 및 CSV 저장 완료!")

--- 전처리 완료 샘플 데이터 확인 (영문 컬럼명 및 기준온도 10도 반영) ---


,Location,Date,Temp_Avg,Temp_Range,Accumulated_Temp
0,강릉,2016-01-01,5.1,6.8,0.0
1,강릉,2016-01-02,9.1,5.7,0.0
2,강릉,2016-01-03,9.0,8.5,0.0
3,강릉,2016-01-04,7.4,8.7,0.0
4,강릉,2016-01-05,3.0,7.3,0.0



전국 전체 지점의 적산온도(기준 10도) 컬럼 추가 및 CSV 저장 완료!
